# 구글 마운트

In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import re
import time
import random
from collections import defaultdict

PITCHER_BOX_DIR = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/pitcher_box"


def parse_ip(ip_str):
    try:
        ip_str = str(ip_str).strip()
        if '.' in ip_str:
            full, partial = ip_str.split('.')
            return int(full) + int(partial) / 3
        else:
            return float(ip_str)
    except:
        return None


In [ ]:

def _strip_name_no(value):
    """투수기록 표의 이름 셀도 타자기록과 동일하게 "홍길동(4)" 형태로 내려오는데,
    그동안은 번호를 안 떼고 그대로 저장해왔다. 번호가 pitcher_box에 남아 있으면
    Streamlit 쪽 이름 검색/매칭이 깨지므로 크롤링 시점에 떼어낸다."""
    if not value:
        return value
    m = re.match(r'^(.*?)\((\d*)\D*$', value.strip())
    return m.group(1).strip() if m else value.strip()


def _parse_pitcher_box_from_soup(soup, game_idx):
    records = []

    hero = soup.select('.hero_sum .team')
    team1 = hero[0].select_one('.txt').get_text(strip=True) if len(hero) > 0 else None
    team2 = hero[1].select_one('.txt').get_text(strip=True) if len(hero) > 1 else None

    sections = soup.select('.section_sumrec')
    for section in sections:
        h4 = section.select_one('h4')
        if not h4 or '투수기록' not in h4.get_text():
            continue

        def_team = h4.get_text(strip=True).replace('투수기록', '').strip()
        table = section.select_one('table[summary="투수기록"]')
        if not table:
            continue

        rows = table.select('tbody tr')
        for row in rows:
            if 'sum' in row.get('class', []):
                continue
            cells = row.find_all(['th', 'td'])
            if len(cells) < 16:
                continue
            try:
                record = {
                    'game_idx': game_idx,
                    'team1':    team1,
                    'team2':    team2,
                    'def_team': def_team,
                    'pitcher':  _strip_name_no(cells[0].get_text(strip=True)),
                    'entry':    cells[1].get_text(strip=True),
                    'result':   cells[2].get_text(strip=True),
                    'W':        cells[3].get_text(strip=True),
                    'L':        cells[4].get_text(strip=True),
                    'IP_raw':   cells[5].get_text(strip=True),
                    'BF':       cells[6].get_text(strip=True),
                    'pitches':  cells[7].get_text(strip=True),
                    'AB':       cells[8].get_text(strip=True),
                    'H':        cells[9].get_text(strip=True),
                    'HR':       cells[10].get_text(strip=True),
                    'BB_HBP':   cells[11].get_text(strip=True),
                    'K':        cells[12].get_text(strip=True),
                    'R':        cells[13].get_text(strip=True),
                    'ER':       cells[14].get_text(strip=True),
                    'ERA':      cells[15].get_text(strip=True),
                }
                records.append(record)
            except Exception as e:
                print(f"  [{game_idx}] 행 파싱 오류: {e}")
                continue

    return records


def parse_pitcher_box(game_idx, session=None):
    url = f"https://www.korea-baseball.com/game/record_detail?game_idx={game_idx}"

    try:
        if session:
            res = session.get(url, timeout=10)
        else:
            res = requests.get(url, timeout=10)
        res.raise_for_status()
    except Exception as e:
        print(f"  [{game_idx}] 요청 실패: {e}")
        return []

    soup = BeautifulSoup(res.text, 'html.parser')
    return _parse_pitcher_box_from_soup(soup, game_idx)


def get_game_year_from_soup(soup):
    tournament_elem = soup.find('dl', class_='game_name')
    if not tournament_elem:
        return None
    date_info = tournament_elem.find('dd').get_text(strip=True)
    date_str = date_info.split()[0]
    try:
        return int(date_str.split('.')[0])
    except (ValueError, IndexError):
        return None


def _format_pitcher_box_df(records):
    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)
    df['IP'] = df['IP_raw'].apply(parse_ip)

    int_cols = ['W', 'L', 'BF', 'pitches', 'AB', 'H', 'HR', 'BB_HBP', 'K', 'R', 'ER']
    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    df['ERA'] = pd.to_numeric(df['ERA'], errors='coerce')

    col_order = ['game_idx', 'team1', 'team2', 'def_team', 'pitcher',
                 'entry', 'result', 'W', 'L', 'IP_raw', 'IP',
                 'BF', 'pitches', 'AB', 'H', 'HR', 'BB_HBP', 'K', 'R', 'ER', 'ERA']
    return df[[c for c in col_order if c in df.columns]]


def _save_pitcher_box(records, output_path):
    df = _format_pitcher_box_df(records)
    if df.empty:
        return df
    df.to_excel(output_path, index=False)
    return df


def recrawl_pitcher_box(game_ids, output_dir=None, max_retry=3):
    """
    특정 game_id(들)의 투수 박스스코어를 다시 크롤링해서 pitcher_box_{year}.xlsx에 덮어쓴다.
    이미 파일에 해당 game_id 행이 있으면 전부 삭제한 뒤, 새로 크롤링한 기록을
    마지막 행 다음부터 append한다 (연도는 크롤링한 경기 날짜로 자동 판별).
    """
    if isinstance(game_ids, int):
        game_ids = [game_ids]
    game_ids = [int(gid) for gid in game_ids]

    if output_dir is None:
        output_dir = PITCHER_BOX_DIR
    os.makedirs(output_dir, exist_ok=True)

    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    })

    records_by_year = defaultdict(list)
    failed = []

    for i, game_id in enumerate(game_ids, 1):
        url = f"https://www.korea-baseball.com/game/record_detail?game_idx={game_id}"
        soup = None

        for attempt in range(1, max_retry + 1):
            try:
                res = session.get(url, timeout=10)
                res.raise_for_status()
                soup = BeautifulSoup(res.text, 'html.parser')
                break
            except Exception as e:
                if attempt < max_retry:
                    retry_wait = random.uniform(9.0, 11.0)
                    print(f"  [{i}/{len(game_ids)}] {game_id} → 요청 실패 ({e}) / 재시도 {attempt}/{max_retry} ({retry_wait:.1f}초 후)")
                    time.sleep(retry_wait)
                else:
                    print(f"  [{i}/{len(game_ids)}] {game_id} → 최종 요청 실패: {e}")

        if soup is None:
            failed.append(game_id)
        else:
            year = get_game_year_from_soup(soup)
            pitcher_records = _parse_pitcher_box_from_soup(soup, game_id)

            if year is None or not pitcher_records:
                print(f"  [{i}/{len(game_ids)}] {game_id} → 연도/기록 파싱 실패, 스킵")
                failed.append(game_id)
            else:
                records_by_year[year].extend(pitcher_records)
                print(f"  [{i}/{len(game_ids)}] {game_id} ({year}년) → {len(pitcher_records)}명")

        if i < len(game_ids):
            time.sleep(random.uniform(9.0, 11.0))

    for year, new_records in sorted(records_by_year.items()):
        output_path = os.path.join(output_dir, f'pitcher_box_{year}.xlsx')
        recrawled_ids = {r['game_idx'] for r in new_records}

        if os.path.exists(output_path):
            existing_df = pd.read_excel(output_path)
            before = len(existing_df)
            existing_df = existing_df[~existing_df['game_idx'].isin(recrawled_ids)]
            removed = before - len(existing_df)
            print(f"  [{year}] 기존 {before}행 중 game_id {sorted(recrawled_ids)} 관련 {removed}행 삭제")
        else:
            existing_df = pd.DataFrame()

        new_df = _format_pitcher_box_df(new_records)
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        combined_df.to_excel(output_path, index=False)
        print(f"  [{year}] 저장 완료: {output_path} (총 {len(combined_df)}행)")

    print("\n" + "=" * 60)
    print(f"[투수 박스 재크롤링] 완료 → 성공: {len(game_ids) - len(failed)} / 실패: {len(failed)}")
    if failed:
        print(f"  실패한 game_id: {failed}")
    print("=" * 60)

    return failed
